# Data distribution analysis

Inspects what's actually going into training/validation/testing: how many `.zarr` cases are
in each split, the distribution of each boundary-condition (global) parameter per split, and
the distribution of surface mesh size and the target temperature field per split.

This is a sanity check, not part of the training pipeline: e.g. it's the kind of thing that
would have caught "the val set only covers half the operating range train does" long before
it shows up as confusing validation-loss behavior.

**Reads `.zarr` arrays directly** (via the `zarr` package only) -- it does not import
PhysicsNeMo, so it runs on any machine, GPU or not, independent of whatever environment issues
affect `train.py`/`test.py`. Needs `pandas` (not otherwise required by this project) and
`jupyter`/`ipykernel` to run as a notebook: `pip install pandas jupyter`.

**Known limitation**: each case's `global_params_values` array is just N raw numbers -- the
`.zarr` file itself doesn't record which number is which parameter. This notebook labels them
using `conf/config.yaml`'s `variables.global_parameters` order (the same assumption
`utils.get_keys_to_read`'s config-derived fallback makes). If your data's actual per-case
parameter order doesn't match that config order, the labels below will be wrong even though
the values themselves are read correctly -- there's no way to verify the order from the data
alone. See `README.md` / the conversation history around `utils.get_keys_to_read` if this
matters for your dataset.

In [ ]:
from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr
from omegaconf import OmegaConf

%matplotlib inline

## Configuration

Edit this cell to point at different data -- doesn't have to match `conf/config.yaml`.

In [ ]:
# Assumes this notebook lives in notebooks/ directly under the project root.
# Adjust PROJECT_ROOT if you've moved it.
PROJECT_ROOT = Path("..").resolve()

cfg = OmegaConf.load(PROJECT_ROOT / "conf" / "config.yaml")

# Boundary-condition (global parameter) names, in the order conf/config.yaml declares them --
# see the "Known limitation" note above.
BC_NAMES = list(cfg.variables.global_parameters.keys())

# One entry per split to compare. Defaults to this project's current config.yaml paths;
# override freely -- e.g. to check a new data drop before pointing config.yaml at it, or to
# compare two different dataset versions side by side under different split names.
SPLITS = {
    "train": PROJECT_ROOT / cfg.data.input_dir,
    "val": PROJECT_ROOT / cfg.data.input_dir_val,
    "test": PROJECT_ROOT / cfg.eval.test_path,
}

# Cap cases read per split (None = all). Useful for a quick look at a very large dataset.
MAX_CASES_PER_SPLIT = None

# Consistent colors across all plots below.
SPLIT_COLORS = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green"}

print("Boundary condition order:", BC_NAMES)
for name, path in SPLITS.items():
    print(f"  {name}: {path}  (exists: {path.exists()})")

## Load per-case summaries

One row per `.zarr` case: its boundary conditions, mesh size, and target-field stats.

In [ ]:
def summarize_case(case_path: Path, bc_names: list) -> dict:
    """Read one case's .zarr group and return a flat dict of summary stats."""
    group = zarr.open_group(str(case_path), mode="r")
    row = {"case": case_path.stem}

    bc_values = np.asarray(group["global_params_values"]).flatten()
    if len(bc_values) != len(bc_names):
        row["_bc_count_mismatch"] = f"{len(bc_values)} values in .zarr vs {len(bc_names)} names in config"
    for name, value in zip(bc_names, bc_values):
        row[name] = float(value)

    # Geometry (STL) mesh size
    row["n_stl_triangles"] = int(np.asarray(group["stl_centers"]).shape[0])

    # Surface solution mesh size and target field (temperature)
    surface_fields = np.asarray(group["surface_fields"]).flatten()
    row["n_surface_points"] = int(surface_fields.shape[0])
    row["temp_mean"] = float(surface_fields.mean())
    row["temp_min"] = float(surface_fields.min())
    row["temp_max"] = float(surface_fields.max())
    row["temp_std"] = float(surface_fields.std())

    return row


def load_split(name: str, dir_path: Path, bc_names: list, max_cases=None) -> pd.DataFrame:
    """Summarize every *.zarr case in dir_path into one DataFrame row each."""
    if not dir_path.exists():
        print(f"[{name}] directory does not exist, skipping: {dir_path}")
        return pd.DataFrame()

    case_paths = sorted(p for p in dir_path.iterdir() if p.suffix == ".zarr")
    if max_cases is not None:
        case_paths = case_paths[:max_cases]
    if not case_paths:
        print(f"[{name}] no .zarr cases found in: {dir_path}")
        return pd.DataFrame()

    rows = []
    for case_path in case_paths:
        try:
            row = summarize_case(case_path, bc_names)
        except Exception as exc:
            print(f"[{name}] failed to read {case_path.name}: {exc}")
            continue
        row["split"] = name
        rows.append(row)

    print(f"[{name}] loaded {len(rows)} of {len(case_paths)} case(s)")
    return pd.DataFrame(rows)

In [ ]:
df = pd.concat(
    [load_split(name, path, BC_NAMES, MAX_CASES_PER_SPLIT) for name, path in SPLITS.items()],
    ignore_index=True,
)

if "_bc_count_mismatch" in df.columns and df["_bc_count_mismatch"].notna().any():
    print("WARNING: some cases have a different number of global_params_values than "
          "BC_NAMES expects -- BC labels below are unreliable for those cases:")
    print(df.loc[df["_bc_count_mismatch"].notna(), ["split", "case", "_bc_count_mismatch"]])

df.head()

## Case counts per split

In [ ]:
counts = df["split"].value_counts().reindex(SPLITS.keys())
ax = counts.plot(kind="bar", figsize=(5, 4), color=[SPLIT_COLORS.get(s, "tab:gray") for s in counts.index])
ax.set_ylabel("Number of cases")
ax.set_title("Case count per split")
for i, v in enumerate(counts.values):
    ax.text(i, v, str(int(v)) if pd.notna(v) else "0", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Boundary condition distributions by split

Marginal distribution of each boundary-condition parameter, overlaid per split. Look for a split whose range doesn't overlap the others -- that's a coverage gap, not just noise.

In [ ]:
ncols = 3
nrows = -(-len(BC_NAMES) // ncols)  # ceil div
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

for i, bc in enumerate(BC_NAMES):
    ax = axes[i // ncols][i % ncols]
    for split_name in SPLITS:
        values = df.loc[df["split"] == split_name, bc].dropna() if bc in df.columns else pd.Series(dtype=float)
        if len(values) == 0:
            continue
        ax.hist(values, bins=20, alpha=0.5, label=split_name, color=SPLIT_COLORS.get(split_name))
    ax.set_title(bc)
    ax.set_xlabel(bc)
    ax.set_ylabel("count")
    ax.legend()

for j in range(len(BC_NAMES), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")

fig.suptitle("Boundary condition distributions by split", y=1.02)
plt.tight_layout()
plt.show()

## Boundary condition summary statistics

In [ ]:
df.groupby("split")[BC_NAMES].describe().T

## Pairwise boundary-condition coverage

Marginals (above) can each look fine individually while the *joint* coverage still differs between splits -- e.g. train covering the full square of two parameters while val only covers the diagonal. This is what marginal histograms alone can hide.

In [ ]:
pairs = list(combinations(BC_NAMES, 2))
ncols = 3
nrows = -(-len(pairs) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for i, (x, y) in enumerate(pairs):
    ax = axes[i // ncols][i % ncols]
    for split_name in SPLITS:
        sub = df[df["split"] == split_name]
        ax.scatter(sub[x], sub[y], alpha=0.5, s=15, label=split_name, color=SPLIT_COLORS.get(split_name))
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(fontsize=8)

for j in range(len(pairs), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")

fig.suptitle("Pairwise boundary-condition coverage by split", y=1.02)
plt.tight_layout()
plt.show()

## Mesh size distributions

Surface solution point count and STL triangle count per case. A split with systematically smaller/larger meshes than the others can behave differently under `model.surface_points_sample`/`model.geom_points_sample` sampling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for split_name in SPLITS:
    sub = df[df["split"] == split_name]
    axes[0].hist(sub["n_surface_points"], bins=20, alpha=0.5, label=split_name, color=SPLIT_COLORS.get(split_name))
    axes[1].hist(sub["n_stl_triangles"], bins=20, alpha=0.5, label=split_name, color=SPLIT_COLORS.get(split_name))

axes[0].set_title("Surface solution points per case")
axes[0].set_xlabel("n_surface_points")
axes[1].set_title("STL triangles per case")
axes[1].set_xlabel("n_stl_triangles")
for ax in axes:
    ax.set_ylabel("count")
    ax.legend()

plt.tight_layout()
plt.show()

## Target field (surface temperature) distribution by split

Per-case mean/min/max temperature. This is the training target -- a systematic shift here between splits (e.g. val only covering a narrower or hotter range than train) directly affects how meaningful the validation loss is as a proxy for real performance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for stat, ax in zip(["temp_mean", "temp_min", "temp_max"], axes):
    for split_name in SPLITS:
        sub = df[df["split"] == split_name]
        ax.hist(sub[stat], bins=20, alpha=0.5, label=split_name, color=SPLIT_COLORS.get(split_name))
    ax.set_title(f"Per-case {stat}")
    ax.set_xlabel(stat)
    ax.set_ylabel("count")
    ax.legend()
plt.tight_layout()
plt.show()

## Which boundary condition actually predicts the per-case baseline?

Relevant if you're using (or considering) `scripts/rebase_surface_temperature.py` to subtract
a boundary condition from `surface_fields` before training (see README "If relative-L2 looks
good but predicted-vs-true shows no diagonal trend"). That script needs a single BC that's
actually the dominant driver of each case's baseline temperature -- guessing wrong (e.g. only
`Temp_inlet` when a secondary stream like `Temp_inlet_A`/`Temp_inlet_B` actually dominates)
leaves a large baseline mismatch behind, which the recomputed scaling-factor summary alone
can't distinguish from genuine spatial variation.

Below: correlation of each BC with each case's mean surface temperature, pooled across all
loaded splits. The strongest correlate is the best single-BC reference candidate; if none of
them correlates strongly, a single-BC subtraction won't fully remove the baseline shift and
`temp_mean` is driven by some combination of BCs (or something not in `global_params_values`
at all).

In [ ]:
correlations = pd.Series(
    {bc: df[bc].corr(df["temp_mean"]) for bc in BC_NAMES}, name="corr_with_temp_mean"
).sort_values(key=np.abs, ascending=False)
print(correlations.to_string())

best_bc = correlations.index[0]
print(f"\nStrongest single-BC correlate: {best_bc} (r={correlations.iloc[0]:.3f})")

# How much of temp_mean's case-to-case spread does a linear combination of ALL BCs explain?
# If this R^2 is much higher than the best single correlation above, no single BC is an
# adequate reference on its own -- the baseline is driven by a combination.
X = np.column_stack([df[bc].to_numpy() for bc in BC_NAMES] + [np.ones(len(df))])
y = df["temp_mean"].to_numpy()
coeffs, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
residuals = y - X @ coeffs
r2_all_bcs = 1 - np.sum(residuals**2) / np.sum((y - y.mean()) ** 2)
print(f"R^2 of temp_mean ~ linear combination of ALL BCs: {r2_all_bcs:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
correlations.plot(kind="barh", ax=ax, color="tab:purple")
ax.set_xlabel("correlation with per-case temp_mean")
ax.set_title("Which BC best predicts the per-case baseline?")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## Within-case vs. across-case temperature variation

This check exists to catch a specific failure mode: **the relative-L2 metric this project
reports (`utils.metrics_fn_surface`) is `||pred - true|| / ||true||`, not mean-centered.** If
temperature sits on a large near-constant baseline (e.g. ~300 K from `Temp_inlet`) with only
small *spatial* variation across a case's surface, a model that predicts close to the
case-level baseline everywhere -- effectively ignoring `surface_mesh_centers` and just reading
off `global_params_values` -- can score a deceptively low relative L2, because the baseline
dominates `||true||`. That's consistent with "loss/L2 keep improving" alongside "predicted vs.
true is a blob with no diagonal": the aggregate metric looks fine while the actual spatial
pattern is barely resolved.

Below: for each split, **across-case** std of the per-case mean temperature (how much the
baseline shifts case to case) vs. the average **within-case** std (how much temperature
actually varies across one case's surface). If across-case >> within-case, the spatial signal
is a small fraction of what the raw-value relative-L2 metric is sensitive to, and a
baseline-only shortcut is easy for the model to find -- worth checking a mean-centered L2 (or
per-case-normalized loss) instead of concluding the model failed to train.

In [ ]:
variation = pd.DataFrame({
    "n_cases": df.groupby("split").size(),
    "across_case_std_of_mean": df.groupby("split")["temp_mean"].std(),
    "avg_within_case_std": df.groupby("split")["temp_std"].mean(),
}).reindex(SPLITS.keys())
variation["ratio_across_over_within"] = (
    variation["across_case_std_of_mean"] / variation["avg_within_case_std"]
)
print(variation.to_string())

for split_name, row in variation.iterrows():
    if pd.isna(row["ratio_across_over_within"]):
        continue
    ratio = row["ratio_across_over_within"]
    if ratio > 5:
        verdict = (
            f"[{split_name}] ratio={ratio:.1f}: baseline shifts between cases dominate over "
            "within-case spatial variation. A relative-L2 computed on raw (non-centered) "
            "values can look good even if the model barely resolves the spatial pattern -- "
            "worth also tracking a mean-centered L2 for this split."
        )
    elif ratio < 0.5:
        verdict = (
            f"[{split_name}] ratio={ratio:.2f}: within-case spatial variation dominates over "
            "case-to-case baseline shifts -- the raw-value relative-L2 metric should already "
            "be sensitive to spatial-pattern accuracy for this split."
        )
    else:
        verdict = f"[{split_name}] ratio={ratio:.2f}: baseline shift and spatial variation are comparable."
    print(verdict)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(variation))
width = 0.35
ax.bar(x - width / 2, variation["across_case_std_of_mean"], width, label="across-case std of mean", color="tab:red")
ax.bar(x + width / 2, variation["avg_within_case_std"], width, label="avg within-case std", color="tab:blue")
ax.set_xticks(x)
ax.set_xticklabels(variation.index)
ax.set_ylabel("temperature std (data units)")
ax.set_title("Across-case baseline shift vs. within-case spatial variation")
ax.legend()
plt.tight_layout()
plt.show()

## Using this on future splits

Edit the `SPLITS` dict in the Configuration cell above to point at any directory of `.zarr`
cases -- it doesn't have to match `conf/config.yaml`, and split names/keys are arbitrary
(add a fourth split, rename `"test"`, etc.). Then **Kernel -> Restart & Run All**.

If you change `variables.global_parameters` in `conf/config.yaml` (add/remove/reorder
boundary conditions), `BC_NAMES` picks that up automatically on the next run -- no edits
needed here unless you also want to compare against a config.yaml other than this project's
current one.